In [4]:
import numpy as np
from statsmodels.stats.power import TTestPower

def solve_paired_ttest(N=None, shift=None, sigma=None, power=None, alpha=0.05):
    """
    Solves for the missing parameter in a Paired T-Test Power Analysis.
    
    Parameters:
    -----------
    N : int or None
        Number of subjects.
    shift : float or None
        The raw change in PSE (in degrees) you expect or want to detect.
    sigma : float
        The standard deviation of the DIFFERENCE scores (Noise). 
        *Must be provided* (unless you are solving for it, which is rare).
    power : float or None
        Target statistical power (e.g., 0.80).
    alpha : float
        Significance level (default 0.05).
        
    Returns:
    --------
    The calculated value of the parameter set to None.
    """
    
    analysis = TTestPower()
    
    # CASE 1: Solve for Sample Size (N)
    if N is None:
        # Calculate Cohen's d first
        d = shift / sigma
        required_n = analysis.solve_power(effect_size=d, nobs=None, alpha=alpha, power=power, alternative='two-sided')
        print(f"--- SOLVING FOR N ---")
        print(f"Inputs: Shift={shift}°, Noise={sigma}°, Power={power}")
        print(f"Required Sample Size: {np.ceil(required_n):.0f} subjects")
        return np.ceil(required_n)

    # CASE 2: Solve for Minimum Detectable Shift (Sensitivity)
    elif shift is None:
        # We find the required Cohen's d first, then convert to degrees
        required_d = analysis.solve_power(effect_size=None, nobs=N, alpha=alpha, power=power, alternative='two-sided')
        min_shift = required_d * sigma
        print(f"--- SOLVING FOR SHIFT ---")
        print(f"Inputs: N={N}, Noise={sigma}°, Power={power}")
        print(f"Minimum Detectable Shift: {min_shift:.3f} degrees")
        return min_shift

    # CASE 3: Solve for Power
    elif power is None:
        d = shift / sigma
        achieved_power = analysis.solve_power(effect_size=d, nobs=N, alpha=alpha, power=None, alternative='two-sided')
        print(f"--- SOLVING FOR POWER ---")
        print(f"Inputs: N={N}, Shift={shift}°, Noise={sigma}°")
        print(f"Achieved Power: {achieved_power:.3f} ({achieved_power*100:.1f}%)")
        return achieved_power


In [5]:
# SCENARIO A: "I have 8 subjects. How big of a shift can I actually see?"
# (Assuming moderate noise of 1.0 degree)
solve_paired_ttest(N=8, shift=None, sigma=1.0, power=0.8)

--- SOLVING FOR SHIFT ---
Inputs: N=8, Noise=1.0°, Power=0.8
Minimum Detectable Shift: 1.156 degrees


1.1560352719599587

In [6]:
# SCENARIO B: "I want to detect a 0.5 degree shift. How many people do I need?"
# (Assuming moderate noise of 1.0 degree)
solve_paired_ttest(N=None, shift=0.5, sigma=1.0, power=0.8)

--- SOLVING FOR N ---
Inputs: Shift=0.5°, Noise=1.0°, Power=0.8
Required Sample Size: 34 subjects


np.float64(34.0)